# LISA-GCN — HC3 train to GPT-3.5 Mixed clean (0%)

Protocol: train and select the checkpoint using **HC3 train/validation only**. GPT-3.5 Mixed clean is built and evaluated only after training. The fixed decision threshold is 0.5; this notebook does not tune on target labels. The final ZIP includes the seed in its filename.

In [ ]:
from pathlib import Path

SEED = 2025  # Change deliberately for an independent run.
EPOCHS = 10  # Maximum number of training epochs.
PATIENCE = 5  # Stop after this many epochs without validation loss improvement.
BRANCH = 'lisa-prdetect'
REPO_URL = 'https://github.com/tphan1420/PRDetect-Research.git'
WORK = Path('/kaggle/working/lisa_gcn_hc3_to_gpt35_0pct')
REPO = WORK / 'PRDetect-Research'
MODEL_DIR = WORK / 'model'
EVAL_DIR = WORK / 'evaluation_gpt35_0pct'
WORK.mkdir(parents=True, exist_ok=True)
print({'seed': SEED, 'epochs': EPOCHS, 'patience': PATIENCE, 'branch': BRANCH, 'work': str(WORK)})

In [ ]:
import os
if not REPO.exists():
    !git clone --branch {BRANCH} {REPO_URL} "{REPO}"
%cd {REPO}
!git rev-parse HEAD
!pip -q install torch-geometric scikit-learn spacy transformers tqdm
!python -m spacy download en_core_web_sm -q

## Train only on HC3

The trainer rebuilds sentence-aware dependency windows from HC3 raw JSONL. It uses source-fitted length groups, and never reads GPT-3.5 during training or validation.

In [ ]:
TRAIN_JSON = REPO / 'data/original_text/benchmarks/hc3/hc3_train.json'
VAL_JSON = REPO / 'data/original_text/benchmarks/hc3/hc3_val.json'
assert TRAIN_JSON.is_file() and VAL_JSON.is_file(), 'Missing HC3 raw JSONL files'
!python train_lisa_gcn.py --train-json "{TRAIN_JSON}" --val-json "{VAL_JSON}" --output "{MODEL_DIR}" --seed {SEED} --epochs {EPOCHS} --patience {PATIENCE} --effective-batch-size 4 --micro-batch-size 1 --window-chunk-size 16
CHECKPOINT = MODEL_DIR / f'lisa_gcn_seed{SEED}.pt'
assert CHECKPOINT.is_file(), f'Missing checkpoint: {CHECKPOINT}'

## Test only after training: GPT-3.5 Mixed clean (0%)

This is the single target evaluation. The evaluator stores fixed-threshold metrics, a confusion matrix, and every prediction probability.

In [ ]:
GPT35_TEST = REPO / 'data/original_text/benchmarks/gpt35_mixed/gpt3.5_mixed_test_split.json'
assert GPT35_TEST.is_file(), 'Missing GPT-3.5 Mixed clean test JSONL'
!python evaluate_lisa_gcn.py --checkpoint "{CHECKPOINT}" --input-json "{GPT35_TEST}" --output-dir "{EVAL_DIR}" --window-chunk-size 16
import json
metrics = json.loads((EVAL_DIR / 'metrics.json').read_text(encoding='utf-8'))
metrics

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

ZIP_PATH = WORK / f'lisa_gcn_hc3_to_gpt35_0pct_seed{SEED}.zip'
files = [
    CHECKPOINT, EVAL_DIR / 'metrics.json', EVAL_DIR / 'predictions.json',
    REPO / 'configs/lisa_gcn_core.json', REPO / 'train_lisa_gcn.py',
    REPO / 'evaluate_lisa_gcn.py', REPO / 'infer_lisa_gcn.py',
    REPO / 'lisa_gcn/model.py', REPO / 'lisa_gcn/data.py',
    REPO / 'lisa_gcn/windowing.py', REPO / 'lisa_gcn/objective.py'
]
with ZipFile(ZIP_PATH, 'w', ZIP_DEFLATED) as archive:
    for path in files:
        assert path.is_file(), f'Missing artifact: {path}'
        archive.write(path, path.relative_to(REPO) if path.is_relative_to(REPO) else Path('artifacts') / path.name)
print({'zip': str(ZIP_PATH), 'bytes': ZIP_PATH.stat().st_size, 'seed': SEED})